# Access different LLMs & multimodal models (LangChain)

**Before you run**
- Repo root: `uv sync` (core: `langchain-openai`, `langchain-groq`, `langchain-google-genai`, `python-dotenv`).
- Optional extras (install when you reach that section):
  - `pip install python-dotenv langchain_openai langchain-groq  langchain-anthropic langchain-openrouter langchain-huggingface langchain-ollama gTTS google-genai`
- Set API keys in `.env` at repo root.


> **Tip:** Run sections in order. Skip paid blocks if you only have free keys (Groq, Gemini, OpenRouter).


## Paid vs free access paths

**Paid (vendor APIs):** OpenAI, Anthropic — credits + API keys.

**Free / lower-cost (instructor list):**
1. **Groq** — fast inference, free tier for learning
2. **OpenRouter** — routes many models (free tier ~2659 tokens in/out)
3. **Google Gemini API** — limited free models

## LangChain message pattern (all providers)

```
Prompt (input)  →  LLM.invoke(messages)  →  AIMessage (output)
```

Two equivalent ways to pass messages:

1. **Tuple format** (still common): `[("system", "..."), ("human", "...")]`
2. **Message classes:** `SystemMessage(...)`, `HumanMessage(...)`

- **System** = behaviour for the active session (role, language, format).
- **Human** = user input (text, or multimodal content list for image/audio).


In [22]:
from pathlib import Path
import base64
import os

from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()  # reads .env from repo root

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "dogs.png").exists() and (NOTEBOOK_DIR.parent / "19-Day-17May26" / "dogs.png").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent / "19-Day-17May26"
elif not (NOTEBOOK_DIR / "dogs.png").exists() and (NOTEBOOK_DIR / "19-Day-17May26" / "dogs.png").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "19-Day-17May26"

DOGS_IMAGE = NOTEBOOK_DIR / "dogs.png"
AUDIO_MP3 = NOTEBOOK_DIR / "audio.mp3"
OUTPUT_WAV = NOTEBOOK_DIR / "output.wav"
JPG_DEMO_URL = "https://www.coolutils.com/blog/wp-content/uploads/2025/06/jpg.png"


def require_env(name: str) -> str:
    """Load API key from environment; raise a clear error if missing."""
    value = os.getenv(name)
    if not value:
        raise EnvironmentError(f"Set {name} in .env or your environment before this section.")
    os.environ[name] = value
    return value


def encode_file(path: Path) -> str:
    """Base64-encode a local file (image, audio, etc.) for multimodal API payloads."""
    return base64.b64encode(path.read_bytes()).decode("utf-8")


print("Notebook dir:", NOTEBOOK_DIR.resolve())
print("dogs.png:", DOGS_IMAGE.exists(), "| audio.mp3:", AUDIO_MP3.exists())


Notebook dir: /teamspace/studios/this_studio/AI-pioneers-Cohort1/Week-03
dogs.png: True | audio.mp3: True


---
## 1. Text → text (OpenAI, paid)

Pattern: `ChatOpenAI` → `invoke(prompt)` → `.content`


In [7]:
require_env("OPENAI_API_KEY")
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")
message = llm.invoke("What is the capital of India?")
message.content


'New Delhi.'

---
## 2. Text → text (Anthropic Claude, paid)

Package: `langchain-anthropic` · Key: `ANTHROPIC_API_KEY`


In [ ]:
require_env("ANTHROPIC_API_KEY")
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)
response = llm.invoke([HumanMessage(content="What is the capital of India?")])
response.content


---
## 3. Image → text (OpenAI vision, paid)

### OCR / document understanding → multimodal RAG

Vision models accept **base64** `data:image/...;base64,...` URLs or **HTTPS** image URLs.
Modern OCR uses **ViT / transformer** vision models (not only CNN OCR) — important for **MM-RAG** pipelines.


In [9]:
require_env("OPENAI_API_KEY")
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
b64 = encode_file(DOGS_IMAGE)
message_local = HumanMessage(
    content=[
        {"type": "text", "text": "What is in this image?"},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
    ]
)
print("--- Local dogs.png ---")
print(llm.invoke([message_local]).content)


--- Local dogs.png ---
The image features a collection of cartoon-style illustrations of dogs. There are six different dogs shown, each with distinct fur patterns, shapes, and expressions, all appearing cheerful with their tongues out and exaggerated facial features. The background is a solid color, enhancing the playful and vibrant style of the artwork.


In [10]:
message_url = HumanMessage(
    content=[
        {"type": "text", "text": "What is in this image?"},
        {"type": "image_url", "image_url": {"url": JPG_DEMO_URL}},
    ]
)
print("--- JPG demo URL ---")
print(llm.invoke([message_url]).content)


--- JPG demo URL ---
The image contains a graphic related to JPG files. It features the text "What Is a JPG File?" along with an icon representing a JPG file, typically shown as a stylized document with an image symbol and the letters "JPG." The background colors appear to have a gradient effect.


---
## 4. Groq (free tier) — text & vision

**Groq** ≠ **Grok** (xAI). Groq is an inference platform. Key: `GROQ_API_KEY`


### 4a. Text → text


In [12]:
require_env("GROQ_API_KEY")
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")
messages = [
    ("system", "You are a helpful assistant that translates English to Hindi. Translate the user sentence."),
    ("human", "I love programming."),
]
print(llm.invoke(messages).content)


<think>
Okay, the user wants me to translate "I love programming." into Hindi. Let me start by breaking down the sentence. 

First, "I" in Hindi is "मैं" (Ma). That's straightforward. 

Next, "love" translates to "प्यार करता हूँ" (Pyar karta hoon) for a male speaker. If the speaker were female, it would be "प्यार करती हूँ" (Pyar karti hoon). Since the original sentence doesn't specify gender, I'll assume the male form unless told otherwise. 

Then, "programming" is "प्रोग्रामिंग" (Programing) in Hindi. It's a loanword, so the spelling remains similar. 

Putting it all together: "मैं प्रोग्रामिंग का प्यार करता हूँ।" (Ma programing ka pyar karta hoon.) Wait, let me check the word order. In Hindi, the structure is Subject + Object + Verb. So "मैं प्रोग्रामिंग का प्यार करता हूँ" is correct. 

But sometimes, "love programming" can also be translated as "मुझे प्रोग्रामिंग पसंद है" (Mujhe programing pasand hai), which means "I like programming." However, the user specifically said "love," whi

### 4b. Image → text (Groq vision)


In [13]:
require_env("GROQ_API_KEY")
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")
message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe this image in detail."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encode_file(DOGS_IMAGE)}"}},
    ]
)
print(llm.invoke([message]).content)


The image presents a collection of six cartoon-style dog heads, each with distinct features and expressions. The dogs are arranged in two rows of three, set against a teal background.

**Top Row:**

*   The top-left dog has long brown hair, a black nose, and a pink tongue protruding from its mouth. It wears a brown collar.
*   The middle dog has short brown hair, floppy ears, and a pink tongue. It sports a brown collar.
*   The top-right dog boasts curly orange hair, a white muzzle, and a pink tongue. It also wears a brown collar.

**Bottom Row:**

*   The bottom-left dog features short brown hair, floppy ears, and a pink tongue. It wears a black collar.
*   The middle dog has medium-length black and tan hair, a black nose, and an orange tongue. Its ears are floppy.
*   The bottom-right dog has medium-length black and white hair, a black nose, and an orange tongue. Its ears are also floppy.

Each dog's facial expression is unique, with varying eye sizes and shapes, as well as different

---
## 5. OpenRouter (free tier with limits)

Routes to many models (e.g. Claude via OpenRouter). **Free tier:** ~2659 tokens in+out combined.
Browse models: https://openrouter.ai/models?input_modalities=text,image


In [16]:
require_env("OPENROUTER_API_KEY")
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model="anthropic/claude-sonnet-4.5", max_tokens=2048)
messages = [
    ("system", "You are a helpful assistant."),
    ("human", "Can you give a short overview of climate change?"),
]
print(llm.invoke(messages).content)


# Climate Change Overview

**Climate change** refers to long-term shifts in global temperatures and weather patterns, primarily caused by human activities since the mid-20th century.

## Key Points:

**Main Cause:** Burning fossil fuels (coal, oil, gas) releases greenhouse gases like CO₂, which trap heat in Earth's atmosphere—the "greenhouse effect."

**Observable Effects:**
- Rising global temperatures (about 1.1°C warmer since pre-industrial times)
- Melting ice caps and glaciers
- Rising sea levels
- More frequent extreme weather (heatwaves, droughts, floods, storms)
- Shifting ecosystems and wildlife habitats

**Impacts:**
- Threats to food and water security
- Increased health risks
- Economic disruption
- Displacement of communities

**Solutions:** Transitioning to renewable energy, improving energy efficiency, protecting forests, and reducing emissions to limit warming to 1.5-2°C above pre-industrial levels.

The scientific consensus is clear: climate change is real, human-cause

---
## 6. Google Gemini (free tier on select models)

Key: `GOOGLE_API_KEY` · Package: `langchain-google-genai`

Free-tier examples from class: `gemini-2.5-flash-lite`, `gemini-2.5-flash`, `gemini-2.5-flash-preview-tts`


### 6a. Text → text


In [24]:
require_env("GOOGLE_API_KEY")
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
messages = [
    ("system", "You are a helpful assistant that translates English to French."),
    ("human", "I love programming."),
]
print(model.invoke(messages).content)


J'adore programmer.


### 6b. Image → text


In [25]:
require_env("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe this image in detail."},
        {"type": "image_url", "image_url": f"data:image/png;base64,{encode_file(DOGS_IMAGE)}"},
    ]
)
print(llm.invoke([message]).content)


This image displays a grid of six distinct, stylized cartoon dog characters against a solid, muted teal background. The dogs are arranged in two rows of three, each exhibiting unique breeds, fur textures, and highly expressive faces. The art style is vibrant and cartoonish, characterized by bold outlines, exaggerated features, and dynamic poses that convey personality.

Here's a detailed description of each dog:

**Top Row (Left to Right):**

1.  **Scruffy Terrier Mix:** This dog's head and upper chest are visible. It has shaggy, medium-length fur in shades of brown, tan, and dark brown, with lighter fur around its muzzle and chest. Its large, round white eyes with black pupils are wide open, looking directly forward, giving an excited or surprised expression. A large, black, round nose sits prominently on its face. Its mouth is wide open in a happy pant, revealing a pink tongue hanging out and small white teeth. Its medium-length, floppy ears are dark brown. A simple dark collar with 

---
## 7. Provider summary & next steps

| Provider | Access | LangChain package | Modalities in this notebook |
|----------|--------|-------------------|----------------------------|
| OpenAI | Paid | `langchain-openai` | text→text, text→image, image→text |
| Groq | Free tier | `langchain-groq` | text→text, image→text |
| OpenRouter | Free (limited) | `langchain-openrouter` | text→text |
| Gemini | Free tier | `langchain-google-genai` + `google-genai` | text, image, audio in/out |

